In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import transforms
from torch.utils.data import Dataset
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from PIL import Image
from torchsummary import summary

In [2]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
print(torch.cuda.is_available())
print(torch.__version__)

True
2.9.1+cu128


Just forcing use of cuda if available

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

You need to develop the model with following structure:

The shape for input should be (3, 200, 200) (channels first format in PyTorch)\
Next, create a convolutional layer (nn.Conv2d):\
Use 32 filters (output channels)\
Kernel size should be (3, 3) (that's the size of the filter)\
Use 'relu' as activation\
Reduce the size of the feature map with max pooling (nn.MaxPool2d)\
Set the pooling size to (2, 2)\
Turn the multi-dimensional result into vectors using flatten or view\
Next, add a nn.Linear layer with 64 neurons and 'relu' activation\
Finally, create the nn.Linear layer with 1 neuron - this will be the output\
The output layer should have an activation - use the appropriate activation for the binary classification case\
As optimizer use torch.optim.SGD with the following parameters:

torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

In [5]:
class HairClassifier(nn.Module):
    def __init__(
        self,
        in_chan=3,
        out_chan=32,
        kernel_sz=(3, 3),
        pooling_sz=(2, 2),
        neurons=64
    ):
        super(HairClassifier, self).__init__()
        self.conv1 = nn.Conv2d(
            in_channels=in_chan, 
            out_channels=out_chan, 
            kernel_size=kernel_sz
        )
        self.max_pool = nn.MaxPool2d(kernel_size=pooling_sz)
        self.inner = nn.Linear(99*99*32, neurons)
        self.relu = nn.ReLU()
        self.output_layer = nn.Linear(neurons, 1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.max_pool(x)
        x = torch.flatten(x, 1)
        x = self.inner(x)
        x = self.relu(x)
        x = self.output_layer(x)
        return x

#### nn.MSELoss()

Measures the mean squared error (squared L2 norm) between predicted and target values. It is primarily used for regression tasks where the goal is to predict continuous values

As the problem involves classification not using it

#### nn.BCEWithLogitsLoss()

Combines a Sigmoid layer and Binary Cross-Entropy loss in one. It is used for binary classification problems. This function operates on raw logits (unnormalized scores), making it numerically more stable than applying Sigmoid separately

As the problem involves classification this looks like a solid choice

#### nn.CrossEntropyLoss()

Computes the cross-entropy loss between input logits and target class indices. It is used for multi-class classification tasks. The function internally applies LogSoftmax and NLLLoss, so input should contain raw logits, not probabilities.

The problem is classification but it is not multi-class classification, but it may be helpful too but as a second choice or alternative

#### nn.CosineEmbeddingLoss()

Measures the loss given two input tensors and a label (1 or -1) indicating whether the inputs should be similar or dissimilar. It is used in tasks like contrastive learning, face verification, or semantic similarity, where the goal is to learn embeddings with specific angular relationships.

It would not be a good candidate for the classification problem its application is different.

### Question 1
Which loss function you will use?

nn.MSELoss()\
nn.BCEWithLogitsLoss()\
nn.CrossEntropyLoss()\
nn.CosineEmbeddingLoss()

From the previous descriptions the loss function I would be using is nn.BCEWithLogitLoss() as an alternative I would consider using nn.CrossEntropyLoss()

Adding new steps to a function to build a model

In [6]:
def make_model():
    model = HairClassifier()
    model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.002, momentum=0.8)
    return model, optimizer

### Question 2

What's the total number of parameters of the model? You can use torchsummary or count manually.

In [7]:
model, optimizer = make_model()

In [8]:
summary(model, input_size=(3, 200, 200))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 198, 198]             896
         MaxPool2d-2           [-1, 32, 99, 99]               0
            Linear-3                   [-1, 64]      20,072,512
              ReLU-4                   [-1, 64]               0
            Linear-5                    [-1, 1]              65
Total params: 20,073,473
Trainable params: 20,073,473
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.46
Forward/backward pass size (MB): 11.97
Params size (MB): 76.57
Estimated Total Size (MB): 89.00
----------------------------------------------------------------


Total params indicates: 20_073_473

In [9]:
class HairDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.dataset = ImageFolder(root=root_dir, transform=transform)
        self.samples = self.dataset.samples  #(img_path, class_idx)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

Generators and Training

For the next two questions, use the following transformation for both train and test sets:

In [10]:
#input should be (3, 200, 200)
input_size = 200
#ImageNet normalization values
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

#transforms - just resize and normalize
train_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [11]:
#define transformations
train_dataset = HairDataset(
    root_dir="../data/train/",
    transform=train_transforms
)

val_dataset = HairDataset(
    root_dir="../data/test/",
    transform=val_transforms
)

Use batch_size=20

Use shuffle=True for both training, but False for test.

In [12]:
#create dataloaders
batch_sz=20
train_loader = DataLoader(train_dataset, batch_size=batch_sz, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_sz, shuffle=False)

In [13]:
def train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device):
    history = {"acc": [], "loss": [], "val_acc": [], "val_loss": []}

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            #Ensure labels are float and have shape (batch_size, 1)
            labels = labels.float().unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            #For binary classification with BCEWithLogitsLoss, 
            #apply sigmoid to outputs before thresholding for accuracy
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        epoch_loss = running_loss / len(train_dataset)
        epoch_acc = correct_train / total_train
        history["loss"].append(epoch_loss)
        history["acc"].append(epoch_acc)

        model.eval()
        val_running_loss = 0.0
        correct_val = 0
        total_val = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                labels = labels.float().unsqueeze(1)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_running_loss += loss.item() * images.size(0)
                predicted = (torch.sigmoid(outputs) > 0.5).float()
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        val_epoch_loss = val_running_loss / len(val_dataset)
        val_epoch_acc = correct_val / total_val
        history["val_loss"].append(val_epoch_loss)
        history["val_acc"].append(val_epoch_acc)

        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")
        print(f"Val_Loss: {val_epoch_loss:.4f}, Val_Acc: {val_epoch_acc:.4f}")
        #print(type(history), "in")
    #print(type(history), "out_loop")
    return history

In [14]:
model, optimizer = make_model()

criterion = nn.BCEWithLogitsLoss()
num_epochs = 10

scores = train_and_evaluate(
    model, 
    optimizer, 
    train_loader, 
    val_loader, 
    criterion, 
    num_epochs, device
)

Epoch 1/10
Loss: 0.6708, Acc: 0.6775
Val_Loss: 0.6221, Val_Acc: 0.6517
Epoch 2/10
Loss: 0.4730, Acc: 0.7625
Val_Loss: 0.6105, Val_Acc: 0.6766
Epoch 3/10
Loss: 0.3760, Acc: 0.8363
Val_Loss: 0.6331, Val_Acc: 0.6866
Epoch 4/10
Loss: 0.2580, Acc: 0.8950
Val_Loss: 0.5704, Val_Acc: 0.7463
Epoch 5/10
Loss: 0.1527, Acc: 0.9475
Val_Loss: 0.6734, Val_Acc: 0.7612
Epoch 6/10
Loss: 0.1339, Acc: 0.9525
Val_Loss: 0.7329, Val_Acc: 0.7264
Epoch 7/10
Loss: 0.0816, Acc: 0.9788
Val_Loss: 0.6758, Val_Acc: 0.7662
Epoch 8/10
Loss: 0.0279, Acc: 0.9988
Val_Loss: 0.8356, Val_Acc: 0.7662
Epoch 9/10
Loss: 0.0147, Acc: 1.0000
Val_Loss: 0.8380, Val_Acc: 0.7861
Epoch 10/10
Loss: 0.0085, Acc: 1.0000
Val_Loss: 0.8765, Val_Acc: 0.7662


### Question 3

What is the median of training accuracy for all the epochs for this model?

In [15]:
train_acc_median = np.median(scores.get("acc"))
float(train_acc_median)

0.95

### Question 4

What is the standard deviation of training loss for all the epochs for this model?

In [16]:
train_loss_std = np.std(scores.get("loss"))
float(train_loss_std)

0.21161898438054208

Data Augmentation

For the next two questions, we'll generate more data using data augmentations.

Add the following augmentations to your training data generator:

transforms.RandomRotation(50),\
transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1)),\
transforms.RandomHorizontalFlip(),

In [17]:
#input should be (3, 200, 200)
input_size = 200
#ImageNet normalization values
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

#transforms - just resize and normalize
train_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.RandomRotation(50),
    transforms.RandomResizedCrop(input_size, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

### Question 5

Let's train our model for 10 more epochs using the same code as previously.

Note: make sure you don't re-create the model. we want to continue training the model we already started training.

In [18]:
#define transformations
train_dataset = HairDataset(
    root_dir="../data/train/",
    transform=train_transforms
)

val_dataset = HairDataset(
    root_dir="../data/test/",
    transform=val_transforms
)

In [19]:
#create dataloaders
batch_sz=20
train_loader = DataLoader(train_dataset, batch_size=batch_sz, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_sz, shuffle=False)

In [20]:
scores_augment = train_and_evaluate(
    model, 
    optimizer, 
    train_loader, 
    val_loader, 
    criterion, 
    num_epochs, device
)

Epoch 1/10
Loss: 0.7901, Acc: 0.6737
Val_Loss: 0.6070, Val_Acc: 0.7363
Epoch 2/10
Loss: 0.5457, Acc: 0.7275
Val_Loss: 0.5261, Val_Acc: 0.7861
Epoch 3/10
Loss: 0.4951, Acc: 0.7550
Val_Loss: 0.4912, Val_Acc: 0.7662
Epoch 4/10
Loss: 0.4683, Acc: 0.7875
Val_Loss: 0.5190, Val_Acc: 0.7711
Epoch 5/10
Loss: 0.4758, Acc: 0.7725
Val_Loss: 0.4667, Val_Acc: 0.8060
Epoch 6/10
Loss: 0.4772, Acc: 0.7512
Val_Loss: 0.5156, Val_Acc: 0.7811
Epoch 7/10
Loss: 0.4570, Acc: 0.7850
Val_Loss: 0.4949, Val_Acc: 0.7811
Epoch 8/10
Loss: 0.4099, Acc: 0.8200
Val_Loss: 0.4744, Val_Acc: 0.8010
Epoch 9/10
Loss: 0.4131, Acc: 0.8150
Val_Loss: 0.4777, Val_Acc: 0.7910
Epoch 10/10
Loss: 0.4216, Acc: 0.8137
Val_Loss: 0.4992, Val_Acc: 0.7811


What is the mean of test loss for all the epochs for the model trained with augmentations?

In [21]:
test_loss_mean = np.mean(scores_augment.get("val_loss"))
float(test_loss_mean)

0.5071706181660814

### Question 6

What's the average of test accuracy for the last 5 epochs (from 6 to 10) for the model trained with augmentations?

In [22]:
test_acc_mean_last_5 = np.mean(scores_augment.get("val_acc")[5:10])
float(test_acc_mean_last_5)

0.7870646766169155